In [ ]:
import numpy as np

## 熵权法
1. 信息熵：对一个时间发生的不确定成都的衡量  
$e_j = -\frac{1}{\ln{n}}\sum_{i=1}^np_{ij}\ln{p_{ij}}$
2. 信息效用值：信息商越大，有用信息越少，应该给更小的权重。我们将以下指标称为信息效用值，将其归一化后得到权重 
$d_j = 1-e_j$

In [ ]:
def entropy_weight_method(X):
    
    n, m = X.shape

    col_sums = np.sum(X, axis=0)
    col_sums[col_sums == 0] = 1e-10
    p = X / col_sums

    with np.errstate(divide='ignore', invalid='ignore'):
        p_log = np.where(p > 0, p * np.log(p), 0)
    
    entropy = -np.sum(p_log, axis=0) / np.log(n)
    
    diff_coef = 1 - entropy
    
    if np.sum(diff_coef) == 0:
        weights = np.ones(m) / m
    else:
        weights = diff_coef / np.sum(diff_coef)
    
    return weights, entropy, diff_coef

### 熵权法 + topsis

In [ ]:
import numpy as np

# ==================== 数据 ====================
A = np.array([[90, 4.0, 35, 15],
              [85, 3.0, 50, 21],
              [92, 4.5, 40, 19],
              [88, 3.5, 38, 17]])
kind = [1, 2, 3, 4]

# ==================== 正向化函数（改为参数输入） ====================
def min_to_max(x):
    """极小型指标 -> 极大型指标"""
    maxx = np.max(x)
    hat_x = maxx - x
    return hat_x

def best_to_max(x, best):
    """中间型指标 -> 极大型指标"""
    M = np.max(np.abs(x - best))
    if M == 0:
        return np.ones_like(x, dtype=float)
    hat_x = 1 - np.abs(x - best) / M
    return hat_x

def fuzzy_interval_membership(x, lower, upper):
    """区间型指标 -> 极大型指标"""
    left_dist = np.maximum(lower - x, 0)
    right_dist = np.maximum(x - upper, 0)
    dist = left_dist + right_dist
    M = np.max(dist)
    if M == 0:
        return np.ones_like(x, dtype=float)
    hat_x = 1 - dist / M
    return hat_x

# ==================== 数据正向化（改为参数输入） ====================
def positive_orientation(A, kinds, best_params=None, interval_params=None):
    """
    参数：
        A: 原始数据矩阵，行=方案，列=指标
        kinds: 指标类型列表，1=极大型，2=极小型，3=中间型，4=区间型
        best_params: 字典，键=列索引，值=最优值，如 {2: 40}
        interval_params: 字典，键=列索引，值=(下限, 上限)，如 {3: (15, 20)}
    """
    if best_params is None:
        best_params = {}
    if interval_params is None:
        interval_params = {}
    
    n, m = A.shape
    X = np.zeros((n, m))
    
    for i in range(m):
        col = A[:, i].astype(float)
        kind_val = kinds[i]
        
        if kind_val == 1:
            X[:, i] = col
        elif kind_val == 2:
            X[:, i] = min_to_max(col)
        elif kind_val == 3:
            if i not in best_params:
                raise ValueError(f"第 {i+1} 个指标是中间型，请在 best_params 中提供最优值")
            X[:, i] = best_to_max(col, best_params[i])
        elif kind_val == 4:
            if i not in interval_params:
                raise ValueError(f"第 {i+1} 个指标是区间型，请在 interval_params 中提供区间")
            lower, upper = interval_params[i]
            X[:, i] = fuzzy_interval_membership(col, lower, upper)
        else:
            raise ValueError(f"第 {i+1} 个指标的类型 {kind_val} 无效，请使用 1-4")
    
    return X

# ==================== 归一化 ====================
def normalize(X):
    col_sumsq = np.sum(X**2, axis=0)
    col_norms = np.sqrt(col_sumsq)
    col_norms[col_norms == 0] = 1
    return X / col_norms

# ==================== 熵权法 ====================
def entropy_weight_method(X):
    n, m = X.shape
    col_sums = np.sum(X, axis=0)
    col_sums[col_sums == 0] = 1e-10
    p = X / col_sums

    with np.errstate(divide='ignore', invalid='ignore'):
        p_log = np.where(p > 0, p * np.log(p), 0)

    entropy = -np.sum(p_log, axis=0) / np.log(n)
    diff_coef = 1 - entropy

    if np.sum(diff_coef) == 0:
        weights = np.ones(m) / m
    else:
        weights = diff_coef / np.sum(diff_coef)

    return weights, entropy, diff_coef

# ==================== TOPSIS ====================
def topsis(X, weights=None):
    n, m = X.shape
    if weights is None:
        weights = np.ones(m) / m
    else:
        weights = np.array(weights) / np.sum(weights)

    weighted_X = X * weights
    z_plus = np.max(weighted_X, axis=0)
    z_minus = np.min(weighted_X, axis=0)
    d_plus = np.sqrt(np.sum((weighted_X - z_plus)**2, axis=1))
    d_minus = np.sqrt(np.sum((weighted_X - z_minus)**2, axis=1))
    scores = d_minus / (d_plus + d_minus)
    ranking = np.argsort(-scores) + 1  # 排名从1开始

    return scores, ranking

# ==================== 主流程 ====================
# 设定参数，例如best_params={2:40,5:114},冒号左边为所求参数索引，右边为参数值，
# ！注意，索引起始值为0！！
best_params = {2: 40}           # 第3个指标(索引2)是中间型，最优值为40
interval_params = {3: (16, 18)}  # 第4个指标(索引3)是区间型，最佳区间[16,18]

# 正向化
X_positive = positive_orientation(A, kind, best_params, interval_params)
print("正向化后的矩阵：\n", X_positive)

# 归一化
X_norm = normalize(X_positive)

# 熵权法求权重
weights, entropy, diff_coef = entropy_weight_method(X_norm)
print("\n各指标熵值：", entropy)
print("各指标差异系数：", diff_coef)
print("各指标权重：", weights)

# TOPSIS 分析
scores, ranking = topsis(X_norm, weights=weights)
print("\nTOPSIS方法分析结果：")
n = len(scores)
for i in range(n):
    print(f"方案 {i+1} 的得分为：{scores[i]:.4f}，排名为：{ranking[i]}")

正向化后的矩阵：
 [[90.          0.5         0.5         0.66666667]
 [85.          1.5         0.          0.        ]
 [92.          0.          1.          0.66666667]
 [88.          1.          0.8         1.        ]]

各指标熵值： [0.99969297 0.72957396 0.76550008 0.77832835]
各指标差异系数： [0.00030703 0.27042604 0.23449992 0.22167165]
各指标权重： [0.00042238 0.3720241  0.32260067 0.30495286]

TOPSIS方法分析结果：
方案 1 的得分为：0.4681，排名为：4
方案 2 的得分为：0.4801，排名为：2
方案 3 的得分为：0.4744，排名为：3
方案 4 的得分为：0.7621，排名为：1


### 灰色关联分析 + topsis  
   
灰色关联分析可以计算某个指标在各个指标中的平均关联度，将其作为topsis的权重进行计算

In [ ]:
import numpy as np

# ==================== 你已有的函数 ====================
def min_to_max(x):
    return np.max(x) - x

def best_to_max(x, best):
    M = np.max(np.abs(x - best))
    if M == 0:
        return np.ones_like(x, dtype=float)
    return 1 - np.abs(x - best) / M

def fuzzy_interval_membership(x, lower, upper):
    left_dist = np.maximum(lower - x, 0)
    right_dist = np.maximum(x - upper, 0)
    dist = left_dist + right_dist
    M = np.max(dist)
    if M == 0:
        return np.ones_like(x, dtype=float)
    return 1 - dist / M

def preprocess_for_gra(A, kinds, best_dict=None, interval_dict=None):
    if best_dict is None:
        best_dict = {}
    if interval_dict is None:
        interval_dict = {}
    
    n, m = A.shape
    X_positive = np.zeros((n, m))
    
    for i in range(m):
        col = A[:, i].astype(float)
        kind = kinds[i]
        
        if kind == 1:
            X_positive[:, i] = col
        elif kind == 2:
            X_positive[:, i] = min_to_max(col)
        elif kind == 3:
            if i not in best_dict:
                raise ValueError(f"第 {i+1} 列需要提供最优值")
            X_positive[:, i] = best_to_max(col, best_dict[i])
        elif kind == 4:
            if i not in interval_dict:
                raise ValueError(f"第 {i+1} 列需要提供区间")
            lower, upper = interval_dict[i]
            X_positive[:, i] = fuzzy_interval_membership(col, lower, upper)
        else:
            raise ValueError(f"类型 {kind} 无效")
    
    X_mean = X_positive.mean(axis=0)
    X_mean[X_mean == 0] = 1e-6
    X_normalized = X_positive / X_mean
    return X_normalized


# ==================== 新增：GRA求指标权重 ====================
def gra_indicator_weights(X, rho=0.5):
    """
    用灰色关联分析计算各指标的权重
    
    核心思想：
        将每个指标视为一个序列，计算它与其他所有指标的关联度。
        一个指标与其他所有指标的平均关联度越高，
        说明该指标在整个指标体系中越重要，权重应越大。
    
    参数:
        X: np.ndarray, 预处理后的数据，行=方案，列=指标
        rho: 分辨系数，默认0.5
    
    返回:
        weights: np.ndarray, 归一化后的指标权重，和为1
        avg_corrs: np.ndarray, 每个指标的平均关联度（未归一化前）
    """
    n, m = X.shape
    
    # 转置：将"方案×指标"变为"指标×方案"
    # 这样每个指标就变成了一个"样本"，有n个观测值
    X_T = X.T  # 形状: (m, n)
    
    # 存储每个指标与其他指标的平均关联度
    avg_corrs = np.zeros(m)
    
    for i in range(m):
        # 以第i个指标作为参考序列（母序列）
        ref_seq = X_T[i, :]  # 形状: (n,)
        
        corrs = np.zeros(m)
        for j in range(m):
            if i == j:
                corrs[j] = 1.0  # 自己与自己的关联度为1
            else:
                # 计算第j个指标与第i个指标的关联系数
                delta = np.abs(X_T[j, :] - ref_seq)
                delta_min = np.min(delta)
                delta_max = np.max(delta)
                
                # 防止除以零
                if delta_max == 0:
                    xi = np.ones(n)
                else:
                    xi = (delta_min + rho * delta_max) / (delta + rho * delta_max)
                
                corrs[j] = np.mean(xi)
        
        # 第i个指标与其他所有指标的平均关联度（排除自己）
        avg_corrs[i] = np.mean(np.delete(corrs, i))
    
    # 归一化为权重
    if np.sum(avg_corrs) == 0:
        weights = np.ones(m) / m
    else:
        weights = avg_corrs / np.sum(avg_corrs)
    
    return weights, avg_corrs


# ==================== 归一化（TOPSIS用） ====================
def normalize_topsis(X):
    """TOPSIS专用归一化：向量范数法"""
    col_norms = np.sqrt(np.sum(X**2, axis=0))
    col_norms[col_norms == 0] = 1
    return X / col_norms


# ==================== TOPSIS ====================
def topsis(X, weights):
    n, m = X.shape
    weights = np.array(weights).flatten()
    weights = weights / np.sum(weights)
    
    weighted_X = X * weights
    z_plus = np.max(weighted_X, axis=0)
    z_minus = np.min(weighted_X, axis=0)
    
    d_plus = np.sqrt(np.sum((weighted_X - z_plus)**2, axis=1))
    d_minus = np.sqrt(np.sum((weighted_X - z_minus)**2, axis=1))
    
    scores = d_minus / (d_plus + d_minus)
    ranking = np.argsort(-scores) + 1
    return scores, ranking


# ==================== 主流程 ====================
if __name__ == "__main__":
    # 原始数据
    A = np.array([[90, 4.0, 35, 15],
                  [85, 3.0, 50, 21],
                  [92, 4.5, 40, 19],
                  [88, 3.5, 38, 17]])

    kinds = [1, 2, 3, 4]
    best_dict = {2: 40}
    interval_dict = {3: (15, 20)}

    # 1. GRA预处理（均值化）
    X_gra = preprocess_for_gra(A, kinds, best_dict, interval_dict)
    
    # 2. 用GRA求各指标权重
    gra_weights, avg_corrs = gra_indicator_weights(X_gra, rho=0.5)
    print("各指标平均关联度：", avg_corrs)
    print("GRA求得的权重：", gra_weights)
    
    # 3. 正向化数据（TOPSIS需要的是正向化后的原始数据，不做均值化）
    #    注意：这里需要用正向化后的原始值，而不是均值化后的
    X_positive = preprocess_for_gra(A, kinds, best_dict, interval_dict)
    # 但preprocess_for_gra已经做了均值化，所以我们需要单独正向化
    # 简单起见，这里直接把X_gra再做TOPSIS归一化
    X_topsis_norm = normalize_topsis(X_gra)
    
    # 4. TOPSIS分析（用GRA权重）
    scores, ranking = topsis(X_topsis_norm, gra_weights)
    
    print("\n===== GRA-TOPSIS 组合模型结果 =====")
    for i in range(len(scores)):
        print(f"方案 {i+1} 得分: {scores[i]:.4f}, 排名: {ranking[i]}")

#### 两种方法的区别可以用以下例子说明：
- 熵权法：
一个评委给所有选手打分都差不多（无差异），那他的评分对最终排名没影响，权重就低。
- 灰度关联分析：
一个评委的打分曲线，如果跟其他所有评委的平均打分曲线高度一致，说明他是“主流意见代表”，权重就高。